# ARIMA Forecasting on Maharashtra Mandi Prices

This notebook demonstrates how the project trains a simple ARIMA model on historical mandi price data. The dataset `maharashtra_monthly_prices.csv` is sourced from [joshi-vipul/maharashtra_mandi](https://github.com/joshi-vipul/maharashtra_mandi) which aggregates Maharashtra State Agricultural Marketing Board bulletins.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
DATA_PATH = Path('..') / 'data' / 'raw' / 'maharashtra_monthly_prices.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
subset = df[(df['Commodity'] == 'Onion') & (df['APMC'] == 'Lasalgaon')]
series = subset.set_index('date')['modal_price'].resample('MS').mean().dropna()
series.tail()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(series.index, series.values, marker='o')
plt.title('Lasalgaon Onion Modal Price (₹/qtl)')
plt.ylabel('₹ per quintal')
plt.grid(True)
plt.show()

In [ ]:
model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12), enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False)
forecast = result.get_forecast(steps=12)
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int(alpha=0.2)
forecast_df = pd.DataFrame({
    'price': forecast_mean,
    'low': forecast_ci.iloc[:, 0],
    'high': forecast_ci.iloc[:, 1],
})
forecast_df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(series.index, series.values, label='Historical')
plt.plot(forecast_df.index, forecast_df['price'], label='Forecast')
plt.fill_between(forecast_df.index, forecast_df['low'], forecast_df['high'], color='orange', alpha=0.3, label='80% CI')
plt.legend()
plt.title('ARIMA Forecast: Lasalgaon Onion Prices')
plt.ylabel('₹ per quintal')
plt.grid(True)
plt.show()